# Prepare Dense121 Pretrained Model

In [1]:
import os
%pwd
os.chdir('../')
%pwd

'd:\\AI ML Projects\\Image-Based-Chest-Cancer-Detection\\backend'

Configuration

In [2]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [3]:
from src.chest_cancer_classification.constants import CONFIG_YAML_PATH, PARAM_YAML_FILE
from src.chest_cancer_classification.utils.common import read_yaml_file, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_YAML_PATH, params_file_path=PARAM_YAML_FILE):
        self.config = read_yaml_file(config_file_path)
        self.params = read_yaml_file(params_file_path)
        create_directories([self.config.artifact_dir])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        params = self.params

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.rootdir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.base_updated_model_path),
            params_image_size=params.IMAGE_SIZE,
            params_learning_rate=params.LEARNING_RATE,
            params_include_top=params.INCLUDE_TOP,
            params_weights=params.WEIGHTS,
            params_classes=params.CLASSES
        )

        return prepare_base_model_config

Main Components

In [4]:
from logger import logging
import sys
import os
import gdown
import zipfile
import tensorflow as tf

In [ ]:
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input, Flatten
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    def get_base_model(self):
        try:
            logging.info("Downloading base model")
            self.base_model = tf.keras.applications.DenseNet121(
                input_shape=self.config.params_image_size,
                include_top=self.config.params_include_top,
                weights=self.config.params_weights
            )

            self.save_model(self.config.base_model_path, self.base_model)

        except Exception as e:
            raise e

    @staticmethod
    def _prepare_full_model(base_model, classes,  learning_rate, input_shape):
        INPUT_SHAPE = tuple(input_shape)

        base_model.trainable = False

        
        resize_and_rescale = tf.keras.Sequential([
            layers.Resizing(INPUT_SHAPE[0], INPUT_SHAPE[1]),
            layers.Rescaling(1./255),
        ])

        augmentation = tf.keras.Sequential([
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomRotation(0.1),
            tf.keras.layers.RandomZoom(0.1),
        ])

        model = tf.keras.models.Sequential([
            Input(shape=INPUT_SHAPE),
            resize_and_rescale,
            augmentation,

            base_model,

            Flatten(),

            # tf.keras.layers.Dense(128, activation = 'relu',  kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
            # # tf.keras.layers.BatchNormalization(),
            # # Dropout(0.3),
            
            # tf.keras.layers.Dense(128, activation = 'relu'),
            tf.keras.layers.BatchNormalization(),
            # Dropout(0.2),

            

            tf.keras.layers.Dense(4, activation="softmax")
        ])

        optimizer = tf.keras.optimizers.AdamW(
            learning_rate=0.00001,
            weight_decay=1e-6
        )

        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

        model.summary()
        return model

    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            base_model=self.base_model,
            classes=self.config.params_classes,
            learning_rate=self.config.params_learning_rate,
            input_shape=tuple(self.config.params_image_size)
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)


    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)